<div align="center">

<img src="https://raw.githubusercontent.com/winstonsmith1897/DantinoX/main/docs/images/dantinox.png" width="150" alt="DantinoX"/>

</div>

# DantinoX · 08 — Retrievers & Embedder Training

<div align="center">

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winstonsmith1897/DantinoX/blob/main/docs/notebooks/08_retrievers_training.ipynb) &nbsp;
[![PyPI](https://img.shields.io/pypi/v/dantinox?color=7c3aed)](https://pypi.org/project/dantinox/) &nbsp;
[![GitHub](https://img.shields.io/badge/GitHub-DantinoX-181717?logo=github)](https://github.com/winstonsmith1897/DantinoX)

</div>

*Train text embedders with DantinoX — unsupervised SimCSE and supervised contrastive InfoNCE on labelled pairs.*

---

**You’ll learn**
- `EmbedderParadigm` — pooling + L2-normalised embeddings
- Unsupervised SimCSE — `dx.train` on raw text
- Supervised pairs — `EmbedderTrainer.fit_pairs`
- Load & embed — `Embedder.from_run(run_dir).embed([...])`
- Plug into FAISS · LangChain · ChromaDB

**Runtime** — GPU (T4 or better)

---

In [1]:
import os

# Single GPU — must be set before JAX initialises
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import jax

print("JAX devices:", jax.devices())   # should show exactly 1 GPU

  warnings.warn(


JAX devices: [CudaDevice(id=0)]


In [ ]:
!pip install -q uv
!uv pip install --system -q -U "dantinox[data,hub,elf,benchmark]" "flax>=0.12,<0.13" "jax[cuda12]"

In [2]:
import dantinox as dx

dx.doctor()   # environment health check — jax/flax/CUDA alignment, GPU visibility


  ████                █    █                █   █
  █   █  ███  ████  █████       ████   ███   █ █ 
  █   █ █   █ █   █   █    █    █   █ █   █   █  
  █   █ █  ██ █   █   █    █    █   █ █   █  █ █ 
  ████   ████ █   █   ██   ███  █   █  ███  █   █

  JAX/Flax transformer library  v0.4.7



DantinoX doctor
  dantinox             0.4.7
  jax                  0.9.2
  jaxlib               0.9.2
  flax                 0.12.6
  optax                0.2.8
  jax-cuda12-plugin    0.10.0
  jax-cuda12-pjrt      0.10.0
  transformers         4.44.2
  datasets             4.8.5
  devices              cuda:0
  ✗ jax-cuda12-plugin 0.10.0 vs jaxlib 0.9.2 — the CUDA plugin must match jaxlib exactly (PJRT errors otherwise); fix: pip install -U "jax[cuda12]"


{'versions': {'dantinox': '0.4.7',
  'jax': '0.9.2',
  'jaxlib': '0.9.2',
  'flax': '0.12.6',
  'optax': '0.2.8',
  'jax-cuda12-plugin': '0.10.0',
  'jax-cuda12-pjrt': '0.10.0',
  'transformers': '4.44.2',
  'datasets': '4.8.5'},
 'problems': ['jax-cuda12-plugin 0.10.0 vs jaxlib 0.9.2 — the CUDA plugin must match jaxlib exactly (PJRT errors otherwise); fix: pip install -U "jax[cuda12]"'],
 'warnings': [],
 'gpu': ['cuda:0'],
 'ok': False}

In [3]:
import numpy as np

import dantinox as dx

---

## 1 · Unsupervised — SimCSE

Train an embedder with the stock `Trainer` using SimCSE (dropout-based positives). Pulls **wikitext-2** from HuggingFace — no local files needed.

In [4]:
# Uses wikitext-2 (HuggingFace) — no local files needed.
# The Trainer downloads and tokenises it once, then caches a .npy mmap file.
#
# dropout=0.1 is REQUIRED: SimCSE encodes the same window twice with different
# dropout masks. With dropout=0.0 the two views are identical and the loss
# collapses (InfoNCE diagonal = 1, off-diagonal = 1 → no gradient).

cfg = dx.ModelConfig(
    dim=128, n_heads=4, head_size=32, num_blocks=4,
    vocab_size=4_096,   # will be overridden by the tokenizer vocab size at fit time
    causal=False,       # bidirectional encoder
    dropout=0.1,        # REQUIRED for SimCSE
    max_context=128,
)
paradigm = dx.EmbedderParadigm(cfg, pooling="mean", temperature=0.05)
print(paradigm)

EmbedderParadigm(pooling='mean', temperature=0.05)


In [5]:
# No local corpus needed — the Trainer pulls wikitext-2 from HuggingFace,
# trains a BPE tokenizer on the fly, and caches everything as a .npy mmap.

train_cfg = dx.TrainingConfig(
    # ── HuggingFace dataset ──────────────────────────────────────────────────
    dataset_source="huggingface",
    dataset_name="wikitext",
    dataset_config="wikitext-2-raw-v1",
    dataset_text_field="text",
    dataset_split="train",
    tokenizer_type="bpe",
    # ── training ─────────────────────────────────────────────────────────────
    lr=3e-4,
    epochs=3,
    batch_size=64,
    val_frac=0.05,
    warmup_steps=100,
    max_train_tokens=500_000,   # cap for a quick Colab demo
)
print(train_cfg)

TrainingConfig(lr=0.0003, batch_size=64, grad_accum=1, epochs=3, warmup_steps=100, lr_schedule='cosine', optimizer='adamw', grad_clip=1.0, seed=42, use_bf16=False, gradient_checkpointing=False, patience=0, eval_iters=20, val_frac=0.05, val_every=1, n_devices=0, tp_size=1, dataset_source='huggingface', dataset_name='wikitext', dataset_config='wikitext-2-raw-v1', dataset_text_field='text', dataset_split='train', max_train_tokens=500000, streaming=False, tokenizer_type='bpe', tokenizer_path=None, init_from='', log_file='training_log.csv')


In [6]:
run_dir = dx.train(paradigm, training_config=train_cfg)
print("Run dir:", run_dir)


  ──────────────────────────────────────────────────────────────
  EmbedderParadigm  ·  bidirectional
  ──────────────────────────────────────────────────────────────
  run dir       runs/20260715_113338
  parameters    1.6 M  (1,579,136)

  ── model ─────────────────────────────────────────────────────
  128-dim  ·  4h×32  ·  4 blocks  ·  vocab=4,096  ·  ctx=128
  MHA  ·  RoPE  ·  RMSNorm  ·  bidirectional  ·  MLP(×4,SwiGLU)

  ── data ──────────────────────────────────────────────────────
  source        wikitext
  tokenizer     bpe  ·  4096 vocab
  tokens        500,000  (train 475,000  ·  val 25,000)

  ── training ──────────────────────────────────────────────────
  optimizer     adamw  ·  lr=3e-04  ·  cosine  ·  warmup=100
  batch         64
  schedule      3 epochs  ·  57 steps/epoch  ·  171 updates
  precision     fp32
  devices       1× GPU

  ──────────────────────────────────────────────────────────────



  ⚠  only 171 optimizer updates — the model will likely be undertrained; lower batch_size or raise epochs


  step 1: JIT compiling (may take 1-3 min on first run)...


  from .autonotebook import tqdm as notebook_tqdm


Epoch 1/3:   0%|          | 0/57 [00:00<?, ?it/s]

...

Epoch 3/3:  82%|████████▏ | 47/57 [00:00<00:00, 56.39it/s, loss=0.0025]

Epoch 3/3:  95%|█████████▍| 54/57 [00:01<00:00, 58.00it/s, loss=0.0025]

  Epoch 3/3  train=0.0058  val=0.0406  1.1s  421.9k tok/s



  ──────────────────────────────────────────────────────────────
  training complete  ·  best val loss = 0.0369
  saved → runs/20260715_113338
  ──────────────────────────────────────────────────────────────



Run dir: runs/20260715_113338


---

## 2 · Supervised — (anchor, positive) pairs

Contrastive **InfoNCE** training with `EmbedderTrainer.fit_pairs`. Positives are consecutive sentences from the same paragraph.

In [7]:
# We build pairs from wikitext-2 directly: consecutive sentence pairs from the
# same paragraph are semantically related → good positives for InfoNCE.
# No separate labelled dataset required.

from datasets import load_dataset

wiki = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")

def extract_pairs(dataset, max_pairs: int = 2_000) -> list[tuple[str, str]]:
    """Consecutive non-empty sentences in the same paragraph = positive pairs."""
    pairs = []
    for row in dataset:
        text = row["text"].strip()
        if not text or text.startswith(" ="):   # skip section headers
            continue
        sentences = [s.strip() for s in text.split(".") if len(s.strip()) > 30]
        for i in range(len(sentences) - 1):
            pairs.append((sentences[i], sentences[i + 1]))
            if len(pairs) >= max_pairs:
                return pairs
    return pairs

pairs = extract_pairs(wiki, max_pairs=2_000)
print(f"{len(pairs)} pairs extracted")
print("Example:", pairs[0])

2000 pairs extracted
Example: ('Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit', 'Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media')


In [8]:
# Reuse the tokenizer trained by the unsupervised step above.
# Load it from the run directory so both modes share the same vocabulary.
from dantinox.utils.tokenizer import load_tokenizer_from_file

tok = load_tokenizer_from_file(f"{run_dir}/tokenizer.json")

cfg_sup = dx.ModelConfig(
    dim=128, n_heads=4, head_size=32, num_blocks=4,
    vocab_size=tok.vocab_size,
    causal=False,
    dropout=0.1,
    max_context=128,
)
paradigm_sup = dx.EmbedderParadigm(cfg_sup, pooling="mean", temperature=0.05)

trainer = dx.EmbedderTrainer(
    paradigm_sup, tok,
    dx.TrainingConfig(lr=2e-4, epochs=5, batch_size=32),
)
run_dir_sup = trainer.fit_pairs(pairs, run_dir="runs/embedder_supervised")
print("Run dir:", run_dir_sup)

Epoch 1/5:   0%|          | 0/63 [00:00<?, ?it/s]

Epoch 1/5:   0%|          | 0/63 [00:30<?, ?it/s, loss=3.8132]

Epoch 1/5:   2%|▏         | 1/63 [00:30<31:59, 30.96s/it, loss=3.8132]

Epoch 1/5:   2%|▏         | 1/63 [00:30<31:59, 30.96s/it, loss=3.5256]

Epoch 1/5:   2%|▏         | 1/63 [00:31<31:59, 30.96s/it, loss=3.5611]

...

Epoch 5/5:  95%|█████████▌| 60/63 [00:01<00:00, 27.29it/s, loss=2.3561]

Epoch 5/5:  95%|█████████▌| 60/63 [00:01<00:00, 27.29it/s, loss=2.6567]

Epoch 5/5:  95%|█████████▌| 60/63 [00:01<00:00, 27.29it/s, loss=1.6235]

Epoch 5/5: 100%|██████████| 63/63 [00:01<00:00, 37.44it/s, loss=1.6235]

Run dir: /ssd1/marco.simoni/VULNERABILITY/NETGROUP/DantinoX/docs/notebooks/runs/embedder_supervised


---

## 3 · Inference — `Embedder`

Load a trained run and embed text with `Embedder.from_run(run_dir).embed([...])`.

In [9]:
embedder = dx.Embedder.from_run(run_dir_sup)
print(f"dim = {embedder.dim}")

# Use real sentences from the wikitext test split
wiki_test = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
sentences = [
    row["text"].strip()
    for row in wiki_test
    if len(row["text"].strip()) > 60 and not row["text"].strip().startswith(" =")
][:20]

queries   = sentences[:3]
documents = sentences[3:13]

q_vecs = embedder.embed(queries)    # [3, D]
d_vecs = embedder.embed(documents)  # [10, D]

# Cosine similarity — L2-normalised vectors → dot product == cosine
sim = q_vecs @ d_vecs.T             # [3, 10]
for i, q in enumerate(queries):
    best = int(sim[i].argmax())
    print(f"\nQ: {q[:80]!r}")
    print(f"→  {documents[best][:80]!r}  (score={sim[i, best]:.3f})")

dim = 128



Q: 'Robert Boulter is an English film , television and theatre actor . He had a gues'
→  'He had a recurring role in 2003 on two episodes of The Bill , as character " Con'  (score=0.853)

Q: 'In 2006 , Boulter starred alongside Whishaw in the play Citizenship written by M'
→  'Boulter starred in two films in 2008 , Daylight Robbery by filmmaker Paris Leont'  (score=0.839)

Q: 'In 2000 Boulter had a guest @-@ starring role on the television series The Bill '
→  'He had a recurring role in 2003 on two episodes of The Bill , as character " Con'  (score=0.662)


---

## 4 · FAISS — fast vector store

Index the embeddings for millisecond nearest-neighbour search.

In [10]:
# !pip install -q faiss-gpu   # or faiss-cpu

import faiss

# IndexFlatIP = inner product (== cosine similarity for L2-normalised vectors)
index = faiss.IndexFlatIP(embedder.dim)
index.add(d_vecs.astype(np.float32))
print(f"Index: {index.ntotal} documents")

# Retrieve top-2 for each query
D, I = index.search(q_vecs.astype(np.float32), k=2)
for i, q in enumerate(queries):
    print(f"\nQ: {q!r}")
    for rank, (score, doc_idx) in enumerate(zip(D[i], I[i])):
        print(f"  [{rank+1}] ({score:.3f}) {documents[doc_idx]!r}")

Index: 10 documents

Q: 'Robert Boulter is an English film , television and theatre actor . He had a guest @-@ starring role on the television series The Bill in 2000 . This was followed by a starring role in the play Herons written by Simon Stephens , which was performed in 2001 at the Royal Court Theatre . He had a guest role in the television series Judge John Deed in 2002 . In 2004 Boulter landed a role as " Craig " in the episode " Teddy \'s Story " of the television series The Long Firm ; he starred alongside actors Mark Strong and Derek Jacobi . He was cast in the 2005 theatre productions of the Philip Ridley play Mercury Fur , which was performed at the Drum Theatre in Plymouth and the Menier Chocolate Factory in London . He was directed by John Tiffany and starred alongside Ben Whishaw , Shane Zaza , Harry Kent , Fraser Ayres , Sophie Stanton and Dominic Hall .'
  [1] (0.853) 'He had a recurring role in 2003 on two episodes of The Bill , as character " Connor Price " . In 2004

---

## 5a · LangChain integration

Drop the embedder into a LangChain vector store.

In [11]:
# !pip install -q langchain langchain-community

from langchain_community.vectorstores import FAISS as LC_FAISS

lc_embed = embedder.as_langchain_embeddings()
store = LC_FAISS.from_texts(documents, embedding=lc_embed)

results = store.similarity_search("How does JAX work?", k=2)
for r in results:
    print(r.page_content)

/tmp/ipykernel_2537788/3216450135.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS as LC_FAISS


Boulter starred in two films in 2008 , Daylight Robbery by filmmaker Paris Leonti , and Donkey Punch directed by Olly Blackburn . Boulter portrayed a character named " Sean " in Donkey Punch , who tags along with character " Josh " as the " quiet brother ... who hits it off with Tammi " . Boulter guest starred on a two @-@ part episode arc " Wounds " in May 2008 of the television series Waking the Dead as character " Jimmy Dearden " . He appeared on the television series Survivors as " Neil " in November 2008 . He had a recurring role in ten episodes of the television series Casualty in 2010 , as " Kieron Fletcher " . He portrayed an emergency physician applying for a medical fellowship . He commented on the inherent difficulties in portraying a physician on television : " Playing a doctor is a strange experience . Pretending you know what you 're talking about when you don 't is very bizarre but there are advisers on set who are fantastic at taking you through procedures and giving yo

---

## 5b · ChromaDB integration

Use the embedder as a ChromaDB embedding function.

In [12]:
# !pip install -q chromadb

import chromadb

client = chromadb.Client()
col = client.create_collection(
    "dantinox_docs",
    embedding_function=embedder.as_chroma_fn(),
)
col.add(
    documents=documents,
    ids=[str(i) for i in range(len(documents))],
)

results = col.query(query_texts=["What is a transformer?"], n_results=2)
for doc in results["documents"][0]:
    print(doc)

Du Fu 's mother died shortly after he was born , and he was partially raised by his aunt . He had an elder brother , who died young . He also had three half brothers and one half sister , to whom he frequently refers in his poems , although he never mentions his stepmother .
The son of a minor scholar @-@ official , his youth was spent on the standard education of a future civil servant : study and memorisation of the Confucian classics of philosophy , history and poetry . He later claimed to have produced creditable poems by his early teens , but these have been lost .


---

## 6 · Fine-tuning a pretrained model

Turn any pretrained AR / Discrete model into an embedder — pass `model=` to `fit_pairs()` to inject the weights.

In [13]:
# Any DantinoX model can be fine-tuned as an embedder without retraining
# from scratch.  Pass model= to fit_pairs() to inject the pretrained weights.

pretrained_run = "runs/my_discrete_run"   # ← your existing run

# Bootstrap a tiny discrete checkpoint so this cell is runnable standalone —
# in your own workflow, point pretrained_run at any existing DantinoX run.
if not os.path.exists(f"{pretrained_run}/config.yaml"):
    corpus_path = "tiny_wiki.txt"
    with open(corpus_path, "w") as f:
        f.write("\n".join(a for a, _ in pairs[:200]))
    disc_paradigm = dx.build("discrete", dim=64, n_heads=2, head_size=32,
                              num_blocks=2, vocab_size=tok.vocab_size,
                              max_context=64)
    dx.train(disc_paradigm, corpus_path, run_dir=pretrained_run,
              epochs=1, batch_size=16, tokenizer_path=f"{run_dir}/tokenizer.json")

pretrained_cfg   = dx.ModelConfig.from_yaml(f"{pretrained_run}/config.yaml")
pretrained_model = dx.load(pretrained_run)

# Wrap in EmbedderParadigm — same architecture, new contrastive loss
ft_paradigm = dx.EmbedderParadigm(pretrained_cfg, pooling="mean", temperature=0.05)

ft_trainer = dx.EmbedderTrainer(
    ft_paradigm, tok,
    dx.TrainingConfig(lr=5e-5, epochs=5, batch_size=16),  # low lr for fine-tuning
)

# model= injects pretrained weights instead of starting from scratch
run_dir_ft = ft_trainer.fit_pairs(
    pairs,
    model=pretrained_model,
    run_dir="runs/embedder_finetuned",
)
print("Fine-tuning done →", run_dir_ft)

embedder_ft = dx.Embedder.from_run(run_dir_ft)
print("dim:", embedder_ft.dim)


  ──────────────────────────────────────────────────────────────
  discrete  ·  bidirectional
  ──────────────────────────────────────────────────────────────
  run dir       runs/my_discrete_run
  parameters    0.4 M  (394,816)

  ── model ─────────────────────────────────────────────────────
  64-dim  ·  2h×32  ·  2 blocks  ·  vocab=4,096  ·  ctx=64
  MHA  ·  RoPE  ·  RMSNorm  ·  bidirectional  ·  MLP(×4,SwiGLU)

  ── data ──────────────────────────────────────────────────────
  source        tiny_wiki.txt
  tokenizer     char  ·  4096 vocab
  tokens        7,991  (train 7,192  ·  val 799)

  ── training ──────────────────────────────────────────────────
  optimizer     adamw  ·  lr=3e-04  ·  cosine  ·  warmup=400
  batch         16
  schedule      1 epoch  ·  7 steps/epoch  ·  7 updates
  precision     fp32
  devices       1× GPU

  ──────────────────────────────────────────────────────────────



  ⚠  only 7 optimizer updates — the model will likely be undertrained; lower batch_size or raise epochs


  step 1: JIT compiling (may take 1-3 min on first run)...


Epoch 1/1:   0%|          | 0/7 [00:00<?, ?it/s]

  vram 0.0 GB used  ·  peak 1.1/30 GB (4%)


...

Epoch 5/5:  90%|█████████ | 113/125 [00:01<00:00, 108.82it/s, loss=2.5915]

Epoch 5/5:  99%|█████████▉| 124/125 [00:01<00:00, 105.24it/s, loss=2.5915]

Epoch 5/5:  99%|█████████▉| 124/125 [00:01<00:00, 105.24it/s, loss=2.6147]

Epoch 5/5: 100%|██████████| 125/125 [00:01<00:00, 103.40it/s, loss=2.6147]

Fine-tuning done → /ssd1/marco.simoni/VULNERABILITY/NETGROUP/DantinoX/docs/notebooks/runs/embedder_finetuned
dim: 64


---

**Recap** — you learned:
- unsupervised SimCSE and supervised InfoNCE embedder training
- plugging DantinoX embedders into FAISS, LangChain, and ChromaDB

**Next →** [09 · Environment & Troubleshooting](09_environment_troubleshooting.ipynb) · [Open in Colab](https://colab.research.google.com/github/winstonsmith1897/DantinoX/blob/main/docs/notebooks/09_environment_troubleshooting.ipynb)